# Module 12 - 실습 3: Send API Fan-out/Fan-in

## 학습 목표
- `Send` API로 Fan-out(병렬 분배)을 구현할 수 있다
- `operator.add` Reducer로 Fan-in(결과 수집)을 구현할 수 있다
- 병렬 실행으로 처리 시간을 단축할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/12-langgraph-advanced.md` (섹션 2.3)

## 개념 요약

```
순차 실행 (기존):            병렬 실행 (Send API):
[A] → [B] → [C] → [합치기]   ┌→ [A] ─┐
총 시간: A+B+C+합            ├→ [B] ─┼→ [합치기]
                             └→ [C] ─┘
                             총 시간: max(A,B,C)+합
```

**핵심 패턴**:
```python
# Fan-out: Send 리스트 반환
def fan_out(state) -> list[Send]:
    return [Send("worker", {**state, "item": x}) for x in items]

# Fan-in: operator.add로 자동 수집
class State(TypedDict):
    results: Annotated[list, operator.add]  # 각 병렬 결과가 합산됨
```

In [ ]:
import sys
sys.path.insert(0, '..')

import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langgraph.constants import Send
from common.fake_llm import FakeLLM

print("Send API 실습 준비 완료")

## 실습 1: 병렬 처리 State 정의

`operator.add`를 사용한 State를 정의하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: DocumentResult TypedDict: doc_id, summary, word_count
# 힌트 2: ParallelState의 results 필드에 Annotated[list, operator.add] 적용

class DocumentResult(TypedDict):
    """개별 문서 분석 결과."""
    doc_id: str
    summary: str
    word_count: int

class ParallelState(TypedDict):
    """병렬 처리 상태."""
    documents: list[dict]                                    # 입력 문서 목록
    # TODO: results 필드를 Annotated[list[DocumentResult], operator.add] 로 정의
    results: list                                            # 임시 - 수정 필요
    final_report: str | None

## 실습 2: Fan-out 함수 구현

각 문서에 대해 독립적인 분석 노드를 생성하는 함수를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: prepare_node는 문서 수를 출력하고 빈 dict 반환
# 힌트 2: fan_out_documents는 각 문서에 Send("analyze_document", {...}) 를 반환
# 힌트 3: analyze_document_node는 state["_current_doc"]에서 문서 가져와 분석

def prepare_node(state: ParallelState) -> dict:
    """문서 목록을 확인합니다."""
    docs = state.get("documents", [])
    print(f"[준비] {len(docs)}개 문서를 병렬 분석합니다.")
    return {}

def fan_out_documents(state: ParallelState) -> list[Send]:
    """각 문서에 대해 독립적인 분석 노드를 생성합니다 (Fan-out)."""
    documents = state.get("documents", [])
    
    if not documents:
        return [Send("generate_report", state)]
    
    sends = []
    for doc in documents:
        # TODO: 각 문서에 대해 Send("analyze_document", {...}) 추가
        # send_state에 _current_doc과 results=[] 포함
        pass
    
    return sends

def analyze_document_node(state: ParallelState) -> dict:
    """개별 문서를 분석합니다 (병렬 실행됨)."""
    doc = state.get("_current_doc", {})
    doc_id = doc.get("id", "unknown")
    content = doc.get("content", "")
    
    print(f"  [분석] 문서 {doc_id} 분석 중...")
    
    # TODO: word_count와 summary 계산 후 results 반환
    # results: [DocumentResult(doc_id=..., summary=..., word_count=...)]
    pass

def generate_report_node(state: ParallelState) -> dict:
    """모든 병렬 분석 결과를 합쳐서 최종 리포트를 생성합니다 (Fan-in)."""
    results = state.get("results", [])
    
    report_lines = [
        f"멀티 문서 분석 리포트",
        f"{'='*40}",
        f"총 분석 문서: {len(results)}개",
        "",
    ]
    
    for result in results:
        report_lines.append(
            f"  [{result.get('doc_id', '?')}] "
            f"{result.get('summary', '')} "
            f"({result.get('word_count', 0)} words)"
        )
    
    total_words = sum(r.get("word_count", 0) for r in results)
    report_lines.append(f"\n총 단어 수: {total_words}")
    
    print(f"[리포트] 생성 완료")
    return {"final_report": "\n".join(report_lines)}

## 실습 3: 병렬 그래프 구성 및 실행

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: StateGraph(ParallelState)로 그래프 생성
# 힌트 2: add_conditional_edges("prepare", fan_out_documents, ["analyze_document", "generate_report"])
# 힌트 3: add_edge("analyze_document", "generate_report") 로 Fan-in

def build_parallel_graph():
    """병렬 문서 분석 그래프."""
    graph = StateGraph(ParallelState)
    
    # TODO: 노드 추가
    # TODO: 엣지 연결 (Fan-out, Fan-in)
    
    return graph.compile()

app = build_parallel_graph()

In [ ]:
# 실행
result = app.invoke({
    "documents": [
        {"id": "DOC-001", "content": "Python은 간결하고 읽기 쉬운 프로그래밍 언어입니다."},
        {"id": "DOC-002", "content": "LangGraph는 LLM 에이전트 워크플로우를 구축하는 프레임워크입니다."},
        {"id": "DOC-003", "content": "체크포인팅은 장애 복구를 위한 핵심 기능입니다."},
    ],
    "results": [],
    "final_report": None,
})

print(result.get("final_report", "리포트가 없습니다"))

In [ ]:
# 검증 셀
assert result.get("final_report") is not None, "final_report가 없습니다"
assert result.get("results") is not None, "results가 없습니다"
assert len(result["results"]) == 3, f"3개 문서가 분석되어야 합니다. 현재: {len(result['results'])}"

# 모든 문서가 분석되었는지 확인
doc_ids = {r.get("doc_id") for r in result["results"]}
assert "DOC-001" in doc_ids, "DOC-001이 분석되지 않았습니다"
assert "DOC-002" in doc_ids, "DOC-002가 분석되지 않았습니다"
assert "DOC-003" in doc_ids, "DOC-003이 분석되지 않았습니다"

print(f"✅ 실습 3 완료! {len(result['results'])}개 문서를 병렬로 분석했습니다.")

## 정리

이번 실습에서 배운 내용:
1. `Send(노드명, 상태)` 리스트를 반환하면 Fan-out(병렬 분배)이 구현된다
2. `Annotated[list, operator.add]`로 각 병렬 노드 결과가 자동으로 합산된다
3. `analyze_document → generate_report` 엣지가 Fan-in을 구현한다

## 다음 모듈
- **실습 4**: `04_streaming.ipynb` - astream 토큰 스트리밍